<a href="https://colab.research.google.com/github/isaac-sun/20NEWS-FL/blob/main/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 20 Newsgroups Federated Learning — Free-Rider Attack Detection

**DistilBERT + LoRA + Per-Class Shapley Value Detection**

This notebook runs 4 federated learning experiments:
1. **Baseline** (no attack)
2. **DFR** — Disguised Free-Rider (Fraboni et al.)
3. **SDFR** — Scaled Delta Free-Rider (Zhu et al.)
4. **AFR** — Advanced Free-Rider (Zhu et al.)

**Runtime**: ~2-4 hours on Colab T4 GPU, ~6-10 hours on CPU.
**Output**: `results/experiment_results.xlsx` + 10 chart groups in `results/plots/`.

## 1. Setup

If you opened this notebook via the **Open in Colab** badge from GitHub, the repository is already cloned — you can skip to Step 2.

If you uploaded this notebook manually, run the cell below to clone.

In [12]:
# @title Setup: Clone Repository (if needed)
# If you opened this via the Colab badge, the repo is already cloned — skip.
# If you uploaded this notebook manually, uncomment and set your repo below.

import os

REPO_NAME = "20NEWS-FL"

# Detect: are we already in the repo directory?
if os.path.basename(os.getcwd()) == REPO_NAME:
    print(f"Already in {REPO_NAME}/ — skipping clone.")
elif os.path.exists(REPO_NAME):
    print(f"{REPO_NAME}/ already exists, pulling latest...")
    %cd {REPO_NAME}
    !git pull
else:
    print("Cloning repository...")
    # Replace with your GitHub username if you forked the repo
    GITHUB_USER = "isaac-sun"  # @param {type:"string"}
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}

print(f"Working directory: {os.getcwd()}")

Already in 20NEWS-FL/ — skipping clone.
Working directory: /content/20NEWS-FL


In [13]:
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 2.49 KiB | 2.49 MiB/s, done.
From https://github.com/isaac-sun/20NEWS-FL
   6f20a60..d86ac73  main       -> origin/main
Updating 6f20a60..d86ac73
Fast-forward
 colab_demo.ipynb | 800 ++++++++++++++++++++++++++++++-------------------------
 1 file changed, 437 insertions(+), 363 deletions(-)


## 2. Install Dependencies

In [14]:
# @title Install Requirements
!pip install -q torch transformers scikit-learn numpy pandas matplotlib seaborn openpyxl tqdm
print("Dependencies installed.")

Dependencies installed.


## 2b. Hugging Face Token (Recommended)

Authenticated requests get **higher rate limits and faster downloads**.

1. Go to https://huggingface.co/settings/tokens and create a token (type: **Read**)
2. In Colab: click the 🔑 **Secrets** icon in the left sidebar
3. Add a secret named `HF_TOKEN` with your token value
4. Toggle the switch to **ON**

The code automatically reads `HF_TOKEN` from Colab Secrets — your token is encrypted and never committed to GitHub.

In [15]:
# @title HF Token Status
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("No HF_TOKEN set — running unauthenticated (still works, just slower).")
except Exception:
    print("Not in Colab environment — using OS environment variables.")

HF_TOKEN loaded from Colab Secrets.


## 3. Verify GPU Availability

In [16]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("Running on CPU — experiments will be slower but functional.")

PyTorch version: 2.11.0+cu128
CUDA available:  True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB


## 4. Run Experiments

This runs all 4 experiments sequentially.  Expect:
- **T4 GPU**: ~30-60 min per experiment
- **CPU only**: ~1-2 hours per experiment

Progress bars and round-level accuracy will be printed live.

In [ ]:
# @title Run All Experiments
# Optional: tweak these before running
QUICK_TEST = False  # @param {type:"boolean"}

import sys
sys.argv = ["main.py"]

if QUICK_TEST:
    # Override config for a fast smoke test (5 rounds, 10 MC samples)
    import main as m
    from config import Config
    import copy, torch, numpy as np
    from utils.seed import set_seed
    from utils.logger import get_logger
    from data.newsgroups import load_newsgroups
    from fl.client import FLClient
    from fl.server import FLServer
    from fl.aggregation import fedavg_aggregate
    from attacks.dfr import dfr_attack, estimate_dfr_sigma
    from attacks.sdfr import sdfr_attack
    from attacks.afr import afr_attack, AFRState
    from contribution.shapley import (
        estimate_round_shapley_per_class, per_class_to_overall,
        compute_class_metrics, _class_weights_from_loader,
    )
    from detection.utility_score import UtilityScoreTracker
    from utils.partition import iid_partition, non_iid_partition
    from utils.export import export_results

    base = m.Config(
        num_clients=10, num_rounds=5, local_epochs=1,
        num_mc_samples=10, batch_size=16, seed=42,
        results_dir="results",
    )

    set_seed(base.seed)
    train_ds, val_ds, test_ds, input_dim, train_labels = load_newsgroups(
        model_name=base.model_name, val_ratio=base.val_ratio,
        max_seq_length=base.max_seq_length,
    )

    experiments = [
        ("quick_baseline", "none"),
        ("quick_dfr", "dfr"),
        ("quick_sdfr", "sdfr"),
        ("quick_afr", "afr"),
    ]

    all_details, all_summaries, all_curves = [], [], {}
    all_pc_records, all_cum_pc_sv = [], {}

    for exp_name, attack_type in experiments:
        cfg = copy.deepcopy(base)
        cfg.experiment_name = exp_name
        cfg.attack_type = attack_type
        (details, summary, accs, losses,
         pc_records, cum_pc_sv) = m.run_experiment(
            cfg, train_ds, val_ds, test_ds, input_dim, train_labels
        )
        all_details.extend(details)
        all_summaries.append(summary)
        all_curves[exp_name] = {"acc": accs, "loss": losses}
        all_pc_records.extend(pc_records)
        all_cum_pc_sv[exp_name] = cum_pc_sv

    # ── Export results ────────────────────────────────────
    import os as _os
    _os.makedirs("results", exist_ok=True)
    filepath = export_results(all_details, all_summaries, "results",
                              per_class_records=all_pc_records)
    print(f"Excel exported: {filepath}")

    # ── Print summary ────────────────────────────────────
    print("\n" + "=" * 80)
    print("QUICK TEST SUMMARY")
    print("=" * 80)
    for s in all_summaries:
        print(f"  {s['experiment_name']:<20s}  "
              f"acc={s['final_global_accuracy']:.4f}  "
              f"SV_h={s['avg_round_shapley_honest']:.6f}")
    print("=" * 80)
else:
    %run main.py

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


06:25:59 [FL] INFO: A100 optimizations: cudnn.benchmark + TF32 + bf16 autocast


INFO:FL:A100 optimizations: cudnn.benchmark + TF32 + bf16 autocast


06:25:59 [FL] INFO: Using device: cuda


INFO:FL:Using device: cuda


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

## 5. Results

In [ ]:
# @title Display Summary
import pandas as pd
import os

xlsx_path = "results/experiment_results.xlsx"
if os.path.exists(xlsx_path):
    df = pd.read_excel(xlsx_path, sheet_name="experiment_summary")
    display(df)
else:
    print("Results file not found. Did the experiment finish?")

# Show plots if they exist
from IPython.display import Image, display as ipy_display
plots_dir = "results/plots"
if os.path.isdir(plots_dir):
    for f in sorted(os.listdir(plots_dir)):
        if f.endswith(".png"):
            print(f"\n### {f}")
            ipy_display(Image(os.path.join(plots_dir, f)))

## 6. Download Results

In [ ]:
# @title Download Results as ZIP
!zip -r results.zip results/ 2>/dev/null
from google.colab import files
files.download("results.zip")